<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/Demo_pyswisseph_v_2_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](http://1.bp.blogspot.com/_5QlanosqxiQ/SXLOLTu7S7I/AAAAAAAAAm8/0r6C7lCxsic/S1600-R/p21-header-003.png) <br>


---

Website - [Parashar21](https://parashar21.blogspot.com)

#Install Swiss Ephemeris <br>
## This is a demonstration of the usage of the Swiss Ephemeris Software in python on the Google Colab Platform
https://pypi.org/project/pyswisseph/  - Python Package Documentation <br>
https://www.astro.com/swisseph/swephinfo_e.htm <br>
https://www.astro.com/swisseph/swephprg.htm - Programmer Documentation <br>
https://pythonhosted.org/pyswisseph/swisseph-module.html

In [1]:
!pip install pyswisseph > /dev/null

In [2]:
#!wget -q http://www.astro.com/ftp/swisseph/ephe/seas_18.se1
#!wget -q http://www.astro.com/ftp/swisseph/ephe/semo_18.se1
#!wget -q http://www.astro.com/ftp/swisseph/ephe/sepl_18.se1

#Download Swiss Ephemeris Files
#!wget  http://www.astro.com/ftp/swisseph/ephe/seas_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/seas_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/semo_18.se1
!wget -q https://github.com/aloistr/swisseph/raw/master/ephe/semo_18.se1
#!wget  http://www.astro.com/ftp/swisseph/ephe/sepl_18.se1
!wget  -q https://github.com/aloistr/swisseph/raw/master/ephe/sepl_18.se1

!mkdir ephe
!mv *.se1 ephe

In [3]:
import swisseph as swe
swe.version

'2.10.03'

In [4]:
#Constants and Flags necessary for this application are set here
#
swe.set_ephe_path('/content/ephe') # set path to ephemeris files
# Calendar : Julian or Gregorian
SE_GREG_CAL = 1
gregflag = SE_GREG_CAL
#
# Ayanamsha type : Lahiri = 1
swe.set_sid_mode(1)  # Lahiri Aynamsha
#
# whether speed will be calculated along with position of planet
SEFLG_SPEED = int(256)
iflag = SEFLG_SPEED
#hsys = P, Placidus
#ascii P = 080
hsysP =  bytes('P', 'utf-8')

#Personal Data
## Data Input

In [5]:
from datetime import datetime
import pytz

In [6]:
# Current time in Asia/Kolkata
now = datetime.now(pytz.timezone("Asia/Kolkata"))

# Convert to UTC
now_utc = now.astimezone(pytz.utc)

# Extract date-time components
year   = now_utc.year
month  = now_utc.month
day    = now_utc.day
hour   = now_utc.hour
minute = now_utc.minute
second = now_utc.second + now_utc.microsecond / 1e6  # keep fractions of seconds

# Convert to Julian Day
jd_ut, jd_et = swe.utc_to_jd(year, month, day, hour, minute, second, swe.GREG_CAL)

print("UTC datetime:", now_utc)
print("Julian Day (UT):", jd_ut)
print("Julian Day (ET):", jd_et)


UTC datetime: 2026-10-23 09:04:56.425171+00:00
Julian Day (UT): 2461336.879231588
Julian Day (ET): 2461336.8784351335


In [7]:
# Date & Time of Birth
# Convert UTC date to Julian Date
#JD = swe.utc_to_jd(1950,9,17,11,0,0,gregflag)
JD = swe.utc_to_jd(year, month, day, hour, minute, second, gregflag)
print (JD[0], JD[1])
# JD[0] is Ephemeris Time
# JD[1] is Universal Time
natalUT = JD[1]
print(natalUT, 'Julian Date in Universal Time')
natalLON = 72.36
natalLAT = 23.48

2461336.879231588 2461336.8784351335
2461336.8784351335 Julian Date in Universal Time


#Calculations

## Without setting SIDEREAL flag

In [8]:
#Ayanamsha
ayanamsha = swe.get_ayanamsa(natalUT)
print('Lahiri Ayanamsha :', ayanamsha)

Lahiri Ayanamsha : 24.23160192276083


In [9]:
#Position of Planets without SETTING SEFLG_SIDEREAL
#
# body 0 = Sun, 1 = Moon, ... 11 = True Node (Rahu)
bLon = []
bRet = []
for body in [0,1,2,3,4,5,6,11]:
    pData = swe.calc_ut(natalUT,body,iflag)
    #print(pData[0])                    --- this is a major change
    #print(pData[1])                    --- this is a major change
    bLon.append(pData[0][0])            # note this correction
    if pData[0][3] >= 0:                # note this correction
        bRet.append('False')
    else :
        bRet.append('True')
for ix in range(len(bLon)) : print(bLon[ix], bRet[ix])

209.97719758714803 False
353.9643919295641 False
230.92394001471592 False
211.22633096414788 True
134.16596032741225 False
143.1926946023162 False
9.865932956272593 True
327.99614008033393 True


In [10]:
#House Position and Ascendants without setting SEFLG_SIDEREAL
# the 13th value is the longitude of Ascendant
#
P = swe.houses(natalUT,natalLAT,natalLON,hsysP)
for ix in range(len(P[0])) : print(P[0][ix])
print('---')
for ix in range(len(P[1])) : print(P[1][ix])
print('---')
print('Asc Long ',P[1][0])

321.6772381942823
0.49668937866908286
34.593651945837856
62.46970916306702
87.05064693785982
112.01521767749398
141.67723819428232
180.4966893786691
214.59365194583785
242.46970916306705
267.0506469378598
292.015217677494
---
321.6772381942823
242.46970916306705
240.3984966040514
163.91793853415197
328.23405262624294
333.02546302456034
256.5745896395026
153.02546302456034
---
Asc Long  321.6772381942823


## With SIDEREAL flag set

In [11]:
#Position of Planetsd using SEFLG_SIDEREAL flag
#
# body 0 = Sun, 1 = Moon, ... 11 = True Node (Rahu)
bLonS = []
bRetS = []
print(iflag)
SEFLG_SIDEREAL = int(64*1024)      #// sidereal positions
iflag |= SEFLG_SIDEREAL
print(iflag)
for body in [0,1,2,3,4,5,6,11]:
    pData = swe.calc_ut(natalUT,body,iflag)
    #print(pData[0])                    --- this is a major change
    #print(pData[1])                    --- this is a major change
    bLonS.append(pData[0][0])            # note this correction
    if pData[0][3] >= 0:                # note this correction
        bRetS.append('False')
    else :
        bRetS.append('True')
print("Change in Longitude for each planet")
for ix in range(len(bLonS)) : print( bLonS[ix],bRetS[ix], bLonS[ix]-bLon[ix] )

256
65792
Change in Longitude for each planet
185.7433844907867 False -24.23381309636133
329.73057883320286 False -24.233813096361246
206.69012691835465 False -24.233813096361274
186.99251786778655 True -24.23381309636133
109.93214723105098 False -24.233813096361274
118.9588815059549 False -24.233813096361303
345.6321198599113 True 335.7661869036387
303.7623269839727 True -24.233813096361246


In [12]:
#House Position and Ascendants without setting SEFLG_SIDEREAL
# the 13th value is the longitude of Ascendant
#
hsysPb = hsysP.decode("utf-8")
print(hsysPb)
PS = swe.houses_ex(natalUT,natalLAT,natalLON,hsysP,iflag)   # the documentation is WRONG, iflag must be the last parameter, not the second
#for ix in range(len(PS[0])) : print(PS[0][ix])
for ix in range(len(PS[1])) : print(PS[1][ix], PS[1][ix]-P[1][ix])
#for ix in range(len(PS[1])) : print(PS[1][0]-P[1][0])

P
297.443425097921 -24.233813096361303
218.23589606670578 -24.233813096361274
240.3984966040514 0.0
139.6841254377907 -24.233813096361274
304.00023952988164 -24.233813096361303
308.79164992819904 -24.233813096361303
232.34077654314135 -24.233813096361274
128.79164992819906 -24.233813096361274


In [13]:
!date

Fri Oct 23 09:04:56 AM UTC 2026


The output generated here matches with the output generated by the Windows program swetest

#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [Finished scrolling? Now try reading a book. ](http://chronos.yantrajaal.com) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)